# Allen Brain Data Explorer

Browse sample images from Allen Brain Institute data sources to choose the best dataset
for fine-tuning a vision model to identify brain structures in Nissl-stained sections.

**Sections:**
1. Allen Mouse Brain Atlas — Nissl reference sections + SVG structure annotations
2. Common Coordinate Framework (CCFv3) — 3D Nissl volume + annotation volume slices
3. Allen Human Brain Atlas — Human Nissl sections
4. Structure Ontology Explorer — Browse the ~700 brain structure hierarchy
5. Side-by-Side Comparison — Compare data sources for the same brain region

**Data access:** All data is fetched from the public Allen Brain Atlas API (`api.brain-map.org`) and
AllenSDK. No authentication required.

**DBFS fallback:** If the Databricks cluster firewall blocks API calls, download data locally and
upload to DBFS. See the fallback cells marked with `[DBFS FALLBACK]`.

## Setup & Installation

In [1]:
# Install dependencies (run once per cluster)
%pip install allensdk requests matplotlib pillow numpy nrrd

ERROR: Could not find a version that satisfies the requirement nrrd (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for nrrd
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from io import BytesIO
from pathlib import Path
import xml.etree.ElementTree as ET
import re

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

API_BASE = "https://api.brain-map.org/api/v2"

# ---- DBFS FALLBACK ----
# Set USE_DBFS = True if the cluster cannot reach api.brain-map.org.
# Then upload data files to the DBFS_ROOT path below.
USE_DBFS = False
DBFS_ROOT = "/dbfs/FileStore/allen_brain_data"

print("Setup complete.")

Setup complete.


## Helper Functions

In [3]:
def fetch_json(url):
    """Fetch JSON from Allen API."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def fetch_image(url):
    """Fetch an image from a URL and return as PIL Image."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return Image.open(BytesIO(response.content))


def fetch_svg_text(url):
    """Fetch SVG text from Allen API."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text


def display_image_grid(images, titles, cols=4, figsize=(16, 10)):
    """Display a grid of PIL images with titles."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten() if rows * cols > 1 else [axes]
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(np.array(img), cmap="gray" if img.mode == "L" else None)
        axes[i].set_title(title, fontsize=9)
        axes[i].axis("off")
    for j in range(len(images), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()


print("Helpers loaded.")

Helpers loaded.


---
## Section 1: Allen Mouse Brain Atlas — Nissl Reference Sections

The Mouse Brain Atlas contains **132 Nissl-stained coronal sections** spanning the entire
mouse brain (C57BL/6J). Each section has expert-drawn SVG annotations mapping every pixel
to one of ~700 brain structures.

- **Nissl reference dataset ID:** `100960033`
- **Atlas ID:** `602630314` (P56 mouse, coronal)
- **Image download:** `{API}/section_image_download/{id}?downsample=N`
- **SVG annotations:** `{API}/svg_download/{id}`

In [4]:
# Fetch all section images in the Nissl reference dataset (100960033)
NISSL_DATASET_ID = 100960033

url = (
    f"{API_BASE}/data/query.json?"
    f"criteria=model::SectionImage,"
    f"rma::criteria,[data_set_id$eq{NISSL_DATASET_ID}],"
    f"rma::options[num_rows$eqall][order$eq'section_number']"
)
result = fetch_json(url)
nissl_sections = result["msg"]

print(f"Total Nissl sections in reference atlas: {len(nissl_sections)}")
print(f"\nFirst section: ID={nissl_sections[0]['id']}, "
      f"dims={nissl_sections[0]['width']}x{nissl_sections[0]['height']}, "
      f"section_number={nissl_sections[0]['section_number']}")
print(f"Last section:  ID={nissl_sections[-1]['id']}, "
      f"dims={nissl_sections[-1]['width']}x{nissl_sections[-1]['height']}, "
      f"section_number={nissl_sections[-1]['section_number']}")

Total Nissl sections in reference atlas: 0


IndexError: list index out of range

In [ ]:
# Download and display 8 evenly-spaced Nissl sections (downsample=4 for speed)
DOWNSAMPLE = 4  # 0=full-res, 4=~16x smaller, good for browsing

n_samples = 8
step = max(1, len(nissl_sections) // n_samples)
sample_indices = list(range(0, len(nissl_sections), step))[:n_samples]

images = []
titles = []
for idx in sample_indices:
    sec = nissl_sections[idx]
    img_url = f"{API_BASE}/section_image_download/{sec['id']}?downsample={DOWNSAMPLE}"
    img = fetch_image(img_url)
    images.append(img)
    titles.append(f"Section {sec['section_number']} (ID: {sec['id']})")

display_image_grid(images, titles, cols=4, figsize=(18, 10))
print(f"Showing {n_samples} of {len(nissl_sections)} Nissl sections at downsample={DOWNSAMPLE}")

In [ ]:
# Show a single section at higher resolution with more detail
# Pick a mid-brain section (roughly hippocampal level)
mid_section = nissl_sections[len(nissl_sections) // 2]
mid_id = mid_section["id"]

# Downsample=2 gives ~4x smaller than full-res (good detail, manageable size)
img_url = f"{API_BASE}/section_image_download/{mid_id}?downsample=2"
img_hires = fetch_image(img_url)

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.imshow(np.array(img_hires))
ax.set_title(
    f"Mid-brain Nissl section (ID: {mid_id}, "
    f"section #{mid_section['section_number']}) — downsample=2\n"
    f"Full-res: {mid_section['width']}×{mid_section['height']} px",
    fontsize=12
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Image size at downsample=2: {img_hires.size[0]}×{img_hires.size[1]} px")
print(f"Full resolution: {mid_section['width']}×{mid_section['height']} px")

### Structure Annotations (SVG Overlays)

Each atlas section has an SVG file with polygonal boundaries for every brain structure.
The SVG contains `<path>` elements with `structure_id` attributes. We can parse these to
create per-pixel segmentation masks for training.

In [ ]:
# Fetch SVG annotation for the mid-brain section
svg_url = f"{API_BASE}/svg_download/{mid_id}"
svg_text = fetch_svg_text(svg_url)

# Parse SVG to count structures
root = ET.fromstring(svg_text)
ns = {"svg": "http://www.w3.org/2000/svg"}

# Find all path elements (structure boundaries)
paths = root.findall(".//svg:path", ns)
if not paths:
    # Try without namespace
    paths = root.findall(".//{http://www.w3.org/2000/svg}path")
if not paths:
    paths = root.findall(".//path")

# Extract structure IDs from the SVG
structure_ids_in_svg = set()
for path in paths:
    sid = path.get("structure_id") or path.get("data-structure-id", "")
    if sid:
        structure_ids_in_svg.add(sid)

print(f"SVG annotation for section {mid_id}:")
print(f"  Total path elements: {len(paths)}")
print(f"  Unique structure IDs: {len(structure_ids_in_svg)}")
print(f"  SVG size: {len(svg_text):,} characters")
print(f"\n  First 500 chars of SVG:\n{svg_text[:500]}")

### Atlas Plates (Annotated Drawings)

The Allen Reference Atlas includes color-coded annotated drawings for each coronal level.
These show the expert-defined structure boundaries overlaid on the Nissl image.

In [ ]:
# Fetch atlas images (annotated plates) for the P56 mouse coronal atlas
ATLAS_ID = 602630314  # P56, coronal

url = (
    f"{API_BASE}/data/query.json?"
    f"criteria=model::AtlasImage,"
    f"rma::criteria,[atlas_data_set_id$eq{ATLAS_ID}],"
    f"rma::options[num_rows$eqall][order$eq'section_number']"
)
atlas_result = fetch_json(url)
atlas_images = atlas_result["msg"]

print(f"Total annotated atlas plates: {len(atlas_images)}")

# Download and display 8 evenly-spaced atlas plates
step = max(1, len(atlas_images) // 8)
plate_indices = list(range(0, len(atlas_images), step))[:8]

plate_images = []
plate_titles = []
for idx in plate_indices:
    plate = atlas_images[idx]
    plate_url = f"{API_BASE}/atlas_image_download/{plate['id']}?downsample=4&annotation=true"
    img = fetch_image(plate_url)
    plate_images.append(img)
    plate_titles.append(f"Plate {plate['section_number']} (ID: {plate['id']})")

display_image_grid(plate_images, plate_titles, cols=4, figsize=(18, 10))
print("Atlas plates show color-coded brain structures overlaid on Nissl reference.")

---
## Section 2: Common Coordinate Framework (CCFv3) — 3D Volumes

The CCFv3 provides 3D reference volumes at 10/25/50 μm resolution:
- **Nissl volume** (`ara_nissl`) — reconstructed 3D Nissl histology
- **Annotation volume** — each voxel labeled with a structure ID
- **Template volume** — average autofluorescence brain

We can slice these at any plane to generate training pairs.

In [ ]:
# Download CCFv3 volumes at 25μm resolution (smaller, faster to explore)
# On Databricks, set USE_DBFS=True and provide pre-downloaded files
from allensdk.core.reference_space_cache import ReferenceSpaceCache

if USE_DBFS:
    import nrrd
    annotation, _ = nrrd.read(f"{DBFS_ROOT}/annotation_25.nrrd")
    template, _ = nrrd.read(f"{DBFS_ROOT}/template_25.nrrd")
    print(f"Loaded from DBFS: annotation {annotation.shape}, template {template.shape}")
    # For structure tree, load from a pre-saved JSON
    rsc = None
else:
    cache_dir = Path("/tmp/allen_ccf_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    rsc = ReferenceSpaceCache(
        resolution=25,
        reference_space_key="annotation/ccf_2017",
        manifest=str(cache_dir / "manifest.json")
    )
    annotation, _ = rsc.get_annotation_volume()
    template, _ = rsc.get_template_volume()

print(f"Annotation volume shape: {annotation.shape} (AP × DV × ML at 25μm)")
print(f"Template volume shape:   {template.shape}")
print(f"Unique structure IDs in annotation: {len(np.unique(annotation))}")
print(f"Annotation dtype: {annotation.dtype}, range: [{annotation.min()}, {annotation.max()}]")

In [ ]:
# Load structure ontology tree
if rsc is not None:
    structure_tree = rsc.get_structure_tree()
    structures = structure_tree.get_structures_by_set_id([167587189])  # Summary structures
    all_structures = structure_tree.get_structures_by_id(list(np.unique(annotation)))
else:
    # Fetch from API directly
    ontology_url = f"{API_BASE}/structure_graph_download/1.json"
    ontology_data = fetch_json(ontology_url)
    all_structures = []
    print("Structure tree loaded from API.")

print(f"Total structures with voxels in annotation volume: {len(np.unique(annotation))}")

# Show a sample of structure names
if rsc is not None:
    sample_structs = all_structures[:15]
    print("\nSample brain structures:")
    for s in sample_structs:
        color_hex = "#{:02x}{:02x}{:02x}".format(*s["rgb_triplet"]) if s.get("rgb_triplet") else "N/A"
        print(f"  ID {s['id']:>6} | {s['acronym']:>8} | {s['name']:<40} | color: {color_hex}")

In [ ]:
# Display coronal slices through the 3D template volume
# The first axis is anterior-posterior (AP)
n_slices = 8
ap_size = template.shape[0]
slice_positions = np.linspace(20, ap_size - 20, n_slices, dtype=int)

fig, axes = plt.subplots(2, n_slices, figsize=(20, 8))

for i, ap in enumerate(slice_positions):
    # Template (grayscale reference)
    axes[0, i].imshow(template[ap, :, :], cmap="gray")
    axes[0, i].set_title(f"AP={ap} ({ap*25}μm)", fontsize=9)
    axes[0, i].axis("off")

    # Annotation (colored by structure ID)
    axes[1, i].imshow(annotation[ap, :, :], cmap="nipy_spectral")
    axes[1, i].set_title(f"Annotation AP={ap}", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Template", fontsize=11)
axes[1, 0].set_ylabel("Annotation", fontsize=11)
plt.suptitle("CCFv3 Coronal Slices — Template (top) vs Annotation (bottom) at 25μm", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Show all three planes (coronal, sagittal, axial) at mid-brain
mid_ap = template.shape[0] // 2
mid_dv = template.shape[1] // 2
mid_ml = template.shape[2] // 2

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

planes = [
    (template[mid_ap, :, :], annotation[mid_ap, :, :], f"Coronal (AP={mid_ap})"),
    (template[:, :, mid_ml], annotation[:, :, mid_ml], f"Sagittal (ML={mid_ml})"),
    (template[:, mid_dv, :], annotation[:, mid_dv, :], f"Axial (DV={mid_dv})"),
]

for i, (tmpl, annot, title) in enumerate(planes):
    axes[0, i].imshow(tmpl, cmap="gray")
    axes[0, i].set_title(f"Template — {title}", fontsize=10)
    axes[0, i].axis("off")

    axes[1, i].imshow(annot, cmap="nipy_spectral")
    axes[1, i].set_title(f"Annotation — {title}", fontsize=10)
    axes[1, i].axis("off")

plt.suptitle("CCFv3 — Three orthogonal planes at mid-brain (25μm)", fontsize=13)
plt.tight_layout()
plt.show()

print("The annotation volume assigns a structure ID to every voxel.")
print("Slicing at any angle generates a training pair: (grayscale image, structure mask).")

In [ ]:
# Detailed view: overlay annotation contours on template for a single coronal slice
from matplotlib.colors import ListedColormap

# Pick a slice at hippocampal level (~AP 200 at 25μm)
ap_hippo = min(200, template.shape[0] - 1)
tmpl_slice = template[ap_hippo, :, :]
annot_slice = annotation[ap_hippo, :, :]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Raw template
axes[0].imshow(tmpl_slice, cmap="gray")
axes[0].set_title("Template (would be Nissl at 10μm)", fontsize=11)
axes[0].axis("off")

# Annotation colored
axes[1].imshow(annot_slice, cmap="nipy_spectral")
axes[1].set_title("Annotation (structure IDs)", fontsize=11)
axes[1].axis("off")

# Overlay: template + semi-transparent annotation
axes[2].imshow(tmpl_slice, cmap="gray")
masked_annot = np.ma.masked_where(annot_slice == 0, annot_slice)
axes[2].imshow(masked_annot, cmap="nipy_spectral", alpha=0.4)
axes[2].set_title("Overlay (template + annotation)", fontsize=11)
axes[2].axis("off")

plt.suptitle(f"CCFv3 Coronal Slice at AP={ap_hippo} (~hippocampal level)", fontsize=13)
plt.tight_layout()
plt.show()

n_structures_in_slice = len(np.unique(annot_slice)) - 1  # exclude 0 (background)
print(f"Structures visible in this slice: {n_structures_in_slice}")

---
## Section 3: Allen Human Brain Atlas — Nissl Sections

The Human Brain Atlas contains Nissl-stained sections from 6 adult donors.
The data is organized by donor, brain region, and plane of section.

In [ ]:
# Query Human Brain Atlas products and datasets
# Product ID 2 = Human Brain Atlas (Allen Human Brain Atlas)
url = (
    f"{API_BASE}/data/query.json?"
    f"criteria=model::SectionDataSet,"
    f"rma::criteria,products[id$eq2],[failed$eqfalse],"
    f"treatments[name$eq'Nissl'],"
    f"rma::include,specimen(donor),plane_of_section,"
    f"rma::options[num_rows$eq50][order$eq'id']"
)
human_result = fetch_json(url)
human_datasets = human_result["msg"]

print(f"Human Nissl section datasets found: {len(human_datasets)}")
for ds in human_datasets[:10]:
    donor_name = ds.get("specimen", {}).get("donor", {}).get("name", "N/A") if ds.get("specimen") else "N/A"
    plane = ds.get("plane_of_section", {}).get("name", "N/A") if ds.get("plane_of_section") else "N/A"
    print(f"  Dataset ID: {ds['id']}, Donor: {donor_name}, Plane: {plane}, "
          f"Sections: {ds.get('section_thickness', 'N/A')}μm thick")

In [ ]:
# Download sample human Nissl sections from the first available dataset
if human_datasets:
    first_human_ds = human_datasets[0]["id"]

    url = (
        f"{API_BASE}/data/query.json?"
        f"criteria=model::SectionImage,"
        f"rma::criteria,[data_set_id$eq{first_human_ds}],"
        f"rma::options[num_rows$eq8][order$eq'section_number']"
    )
    human_sections_result = fetch_json(url)
    human_sections = human_sections_result["msg"]

    human_images = []
    human_titles = []
    for sec in human_sections:
        img_url = f"{API_BASE}/section_image_download/{sec['id']}?downsample=6"
        img = fetch_image(img_url)
        human_images.append(img)
        human_titles.append(
            f"Human sec {sec['section_number']}\n"
            f"{sec['width']}×{sec['height']}px"
        )

    display_image_grid(human_images, human_titles, cols=4, figsize=(18, 10))
    print(f"Showing {len(human_images)} sections from Human Brain Atlas dataset {first_human_ds}")
else:
    print("No human Nissl datasets found. Check API query or try product ID variations.")

---
## Section 4: Structure Ontology Explorer

The Allen Brain Atlas structure ontology is a hierarchical tree of ~700+ brain regions.
Each structure has an ID, name, acronym, RGB color, and parent structure.

In [ ]:
# Fetch the full mouse brain structure ontology
ontology_url = f"{API_BASE}/structure_graph_download/1.json"
ontology_data = fetch_json(ontology_url)

def flatten_ontology(node, depth=0, result=None):
    """Recursively flatten the ontology tree."""
    if result is None:
        result = []
    node["depth"] = depth
    result.append(node)
    for child in node.get("children", []):
        flatten_ontology(child, depth + 1, result)
    return result

flat_structures = flatten_ontology(ontology_data["msg"][0])

print(f"Total structures in ontology: {len(flat_structures)}")
print(f"Max depth: {max(s['depth'] for s in flat_structures)}")
print(f"\nTop-level structures (depth=1):")
for s in flat_structures:
    if s["depth"] == 1:
        color = s.get("color_hex_triplet", "N/A")
        print(f"  {s['acronym']:>8} | {s['name']:<45} | #{color}")

In [ ]:
# Show depth-2 structures (main subdivisions) for a more detailed view
print("Major brain subdivisions (depth=2):")
print(f"{'ID':>8} | {'Acronym':>10} | {'Name':<50} | {'Color':>8}")
print("-" * 85)
for s in flat_structures:
    if s["depth"] == 2:
        color = s.get("color_hex_triplet", "N/A")
        print(f"{s['id']:>8} | {s['acronym']:>10} | {s['name']:<50} | #{color}")

---
## Section 5: Side-by-Side Comparison

Compare data from different sources at a comparable brain level to evaluate
which dataset is best for training.

In [ ]:
# Side-by-side: Mouse Nissl section (API) vs CCF template slice vs CCF annotation
# Use the mid-brain section from Section 1

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

# 1. Mouse Nissl from API (2D section image)
nissl_img_url = f"{API_BASE}/section_image_download/{mid_id}?downsample=4"
nissl_img = fetch_image(nissl_img_url)
axes[0].imshow(np.array(nissl_img))
axes[0].set_title("Mouse Brain Atlas\nNissl Section (2D, API)", fontsize=10)
axes[0].axis("off")

# 2. CCF template coronal slice
axes[1].imshow(template[ap_hippo, :, :], cmap="gray")
axes[1].set_title("CCFv3 Template\nCoronal Slice (3D vol)", fontsize=10)
axes[1].axis("off")

# 3. CCF annotation coronal slice
axes[2].imshow(annotation[ap_hippo, :, :], cmap="nipy_spectral")
axes[2].set_title("CCFv3 Annotation\nStructure Labels (3D vol)", fontsize=10)
axes[2].axis("off")

# 4. CCF overlay
axes[3].imshow(template[ap_hippo, :, :], cmap="gray")
masked = np.ma.masked_where(annotation[ap_hippo, :, :] == 0, annotation[ap_hippo, :, :])
axes[3].imshow(masked, cmap="nipy_spectral", alpha=0.4)
axes[3].set_title("CCFv3 Overlay\nTemplate + Annotation", fontsize=10)
axes[3].axis("off")

plt.suptitle(
    "Data Source Comparison — Same approximate brain level",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

---
## Summary & Dataset Statistics

In [ ]:
print("=" * 80)
print("ALLEN BRAIN DATA SOURCE SUMMARY")
print("=" * 80)
print()
print("1. MOUSE BRAIN ATLAS — Nissl Reference (2D sections)")
print(f"   Sections: {len(nissl_sections)}")
print(f"   Full resolution: ~{nissl_sections[0]['width']}×{nissl_sections[0]['height']} px per section")
print(f"   Annotations: SVG structure boundaries (rasterizable to segmentation masks)")
print(f"   Use: Validation/test set, high-quality 2D training data")
print()
print("2. CCFv3 — 3D Volumes")
print(f"   Template shape: {template.shape} at 25μm")
print(f"   Annotation shape: {annotation.shape} at 25μm")
print(f"   Unique structures: {len(np.unique(annotation))}")
print(f"   Also available at 10μm (with Nissl volume: ara_nissl_10.nrrd)")
print(f"   Use: Generate unlimited 2D training slices at any angle")
print()
print("3. HUMAN BRAIN ATLAS")
print(f"   Nissl datasets found: {len(human_datasets)}")
print(f"   Use: Cross-species generalization (supplementary)")
print()
print("4. STRUCTURE ONTOLOGY")
print(f"   Total structures: {len(flat_structures)}")
print(f"   Hierarchy depth: {max(s['depth'] for s in flat_structures)} levels")
print()
print("=" * 80)
print("RECOMMENDATION")
print("=" * 80)
print()
print("Primary training data:  CCFv3 Nissl volume + Annotation volume (10μm)")
print("  → Perfectly aligned, can slice at any angle for data augmentation")
print("  → Download: allensdk ReferenceSpaceCache or direct NRRD from Allen servers")
print()
print("Validation/test data:   Mouse Brain Atlas Nissl sections (API) + SVG annotations")
print("  → 132 real histological sections with expert annotations")
print("  → Independent from training data (different imaging modality)")
print()
print("Supplementary:          Human Brain Atlas Nissl sections")
print("  → For cross-species evaluation")

---
## [DBFS FALLBACK] Local Download Instructions

If the Databricks cluster firewall blocks `api.brain-map.org`, download the data locally
and upload to DBFS:

```bash
# On your local machine:
mkdir -p allen_brain_data
cd allen_brain_data

# CCFv3 volumes (25μm — smaller, good for exploration)
curl -O https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_25.nrrd
curl -O https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/average_template/average_template_25.nrrd

# CCFv3 volumes (10μm — full resolution for training)
curl -O https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_10.nrrd
curl -O https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/ara_nissl/ara_nissl_10.nrrd

# Upload to DBFS via Databricks UI or CLI:
# databricks fs cp ./allen_brain_data dbfs:/FileStore/allen_brain_data --recursive
```

Then set `USE_DBFS = True` and `DBFS_ROOT = "/dbfs/FileStore/allen_brain_data"` in the Setup cell.